Memory management in Python involves a combination of automatic garbage collection, 
reference counting, and various internal optimizations to efficiently manage memory 
allocation and deallocation. Understanding these mechanisms can help developers write 
more efficient and robust applications.

1. Key Concepts in Python Memory Management
2. Memory Allocation and Deallocation
3. Reference Counting
4. Garbage Collection
5. The gc Module
6. Memory Management Best Practices


#### Reference Counting
Reference counting is the primary method Python uses to manage memory. 
Each object in Python maintains a count of references pointing to it. 
When the reference count drops to zero, the memory occupied by the object is deallocated.

In [1]:
# ================= REFERENCE COUNT EXAMPLE =================
# This program demonstrates how Python keeps track of object references
# using reference counting (a part of memory management).

# Import sys module → provides access to system-specific functions
import sys


# ---------------- CREATE OBJECT ----------------
# Create an empty list and assign it to variable 'a'
a = []

# At this point:
# One reference exists → variable 'a' points to the list object


# ---------------- CHECK REFERENCE COUNT ----------------
# sys.getrefcount(object) returns the number of references to that object
# BUT it includes a temporary reference created when passing the object
# as an argument to the function itself

print(sys.getrefcount(a))


# ---------------- IMPORTANT UNDERSTANDING ----------------
# Expected output: 2

# Why 2 references?
# 1. One reference from variable 'a'
# 2. One temporary reference created when calling getrefcount(a)

# So actual references = 1
# Reported references = 2


# ================= EXTRA EXAMPLES =================

# Add another reference
b = a

# Now:
# 'a' → list
# 'b' → same list

# So total references = 2 (a and b)

print(sys.getrefcount(a))  # Output will be 3
# Explanation:
# 1. 'a'
# 2. 'b'
# 3. temporary reference from getrefcount()


# Remove one reference
del b

# Now only 'a' remains
print(sys.getrefcount(a))  # Output will be 2 again


# ================= KEY CONCEPT =================
# Reference counting is used by Python's garbage collector

# When reference count becomes 0:
# → Object is automatically deleted from memory


# ================= SIMPLE SUMMARY =================
# getrefcount(object) = actual references + 1 (temporary reference)

2
3
2


#### Garbage Collection
Python includes a cyclic garbage collector to handle reference cycles. 
Reference cycles occur when objects reference each other, preventing their reference counts from reaching zero.

In [ ]:
# ================= GARBAGE COLLECTION (GC) EXAMPLE =================
# This program demonstrates how Python's garbage collector works
# and how to control and inspect it using the gc module. 

# ---------------- IMPORT MODULE ----------------
# Import gc module → provides functions to interact with Python's garbage collector
import gc

# ---------------- ENABLE GARBAGE COLLECTION ----------------
# Enable automatic garbage collection
# (Usually enabled by default, but we explicitly enable it here)
gc.enable()

# Check if GC is enabled
print("GC Enabled:", gc.isenabled())


# ---------------- DISABLE GARBAGE COLLECTION ----------------
# This turns OFF automatic garbage collection
# Useful in performance-critical scenarios where you want manual control
gc.disable()

# Check status again
print("GC Enabled after disable:", gc.isenabled())


# ---------------- MANUAL GARBAGE COLLECTION ----------------
# gc.collect() forces Python to run garbage collection immediately
# It returns the number of unreachable objects found and cleaned

collected = gc.collect()

print("Unreachable objects collected:", collected)


# ---------------- GARBAGE COLLECTION STATS ----------------
# gc.get_stats() returns statistics about GC activity
# It shows details for each generation (0, 1, 2)

stats = gc.get_stats()

print("GC Stats:", stats)

# Example output:
# [
#   {'collections': 72, 'collected': 3042, 'uncollectable': 0},
#   {'collections': 6, 'collected': 541, 'uncollectable': 0},
#   {'collections': 6, 'collected': 1737, 'uncollectable': 0}
# ]

# Explanation:
# collections → how many times GC ran
# collected → number of objects freed
# uncollectable → objects that could not be freed


# ---------------- UNREACHABLE OBJECTS ----------------
# gc.garbage contains objects that:
# ❗ Could not be collected (usually due to __del__ or complex references)

print("Uncollectable garbage objects:", gc.garbage)


# ================= IMPORTANT CONCEPT =================
# Python uses TWO memory management techniques:
#
# 1. Reference Counting:
#    → Object is deleted when reference count = 0
#
# 2. Garbage Collector (gc module):
#    → Handles circular references (cycles)



# ================= SIMPLE SUMMARY =================
# gc.enable()  → Turn ON automatic garbage collection
# gc.disable() → Turn OFF automatic garbage collection
# gc.collect() → Manually trigger cleanup
# gc.get_stats() → Get GC performance stats
# gc.garbage → Objects that couldn't be cleaned

GC Enabled: True
GC Enabled after disable: False
Unreachable objects collected: 0
GC Stats: [{'collections': 66, 'collected': 1391, 'uncollectable': 0}, {'collections': 6, 'collected': 541, 'uncollectable': 0}, {'collections': 1, 'collected': 0, 'uncollectable': 0}]
Uncollectable garbage objects: []
Collected after circular reference: 2


##### Memory Management Best Practices
1. Use Local Variables: Local variables have a shorter lifespan and are freed sooner than global variables.
2. Avoid Circular References: Circular references can lead to memory leaks if not properly managed.
3. Use Generators: Generators produce items one at a time and only keep one item in memory at a time, making them memory efficient.
4. Explicitly Delete Objects: Use the del statement to delete variables and objects explicitly.
5. Profile Memory Usage: Use memory profiling tools like tracemalloc and memory_profiler to identify memory leaks and optimize memory usage.

In [1]:
# ================= CIRCULAR REFERENCE HANDLING =================
# This example demonstrates how Python handles circular references
# using the garbage collector (gc module).

# ---------------- IMPORT MODULE ----------------
# Import gc → used to manually control garbage collection
import gc


# ---------------- CLASS DEFINITION ----------------
class MyObject:
    
    # Constructor → called when object is created
    def __init__(self, name):
        self.name = name
        
        # Print message when object is created
        print(f"Object {self.name} created")

    # Destructor → called when object is deleted from memory
    def __del__(self):
        print(f"Object {self.name} deleted")


# ---------------- CREATE OBJECTS ----------------
# Create two objects
obj1 = MyObject("obj1")
obj2 = MyObject("obj2")


# ---------------- CREATE CIRCULAR REFERENCE ----------------
# obj1 references obj2
obj1.ref = obj2

# obj2 references obj1
obj2.ref = obj1

# Now:
# obj1 → obj2
# obj2 → obj1
#
# This creates a circular reference (cycle)


# ---------------- DELETE ORIGINAL REFERENCES ----------------
# Remove direct references from variables
del obj1
del obj2

# At this point:
# ❗ Objects are NOT immediately deleted
# Why?
# Because:
# obj1 still references obj2
# obj2 still references obj1
#
# Their reference count is NOT zero


# ---------------- MANUAL GARBAGE COLLECTION ----------------
# Force Python to run garbage collection
gc.collect()

# This detects the circular reference and cleans it up


# ================= IMPORTANT UNDERSTANDING =================
# 1. Reference counting alone cannot handle cycles
#
# 2. Garbage collector (gc) detects unreachable cycles
#
# 3. gc.collect() forces cleanup immediately


# ================= ⚠️ VERY IMPORTANT NOTE =================
# Objects with __del__() can behave differently in circular references
#
# In some cases:
# Python may NOT automatically delete them
# and they may end up in gc.garbage
#
# This is because Python cannot safely determine
# the order in which destructors should run


# ================= BEST PRACTICE =================
# Avoid circular references OR
# Avoid using __del__() in such cases
#
# Use:
# ✔ weak references (weakref module)
# ✔ context managers (with statement)


# ================= SIMPLE SUMMARY =================
# Circular reference = objects referencing each other
# Reference counting fails → memory not freed
# Garbage collector solves it using gc.collect()

Object obj1 created
Object obj2 created
Object obj1 deleted
Object obj2 deleted


2

In [2]:
## Generrators For Memory Efficiency
# Generators allow you to produce items one at a time, using memory efficiently by onle keeping one item in memory at a time.

def generate_numbers(n):
    for i in range(n):
        yield i

## using the generator
for number in generate_numbers(1000000):
    print(number)
    if number > 10:
        break

0
1
2
3
4
5
6
7
8
9
10
11


In [4]:
# ================= MEMORY PROFILING WITH TRACEMALLOC =================
# This program demonstrates how to track memory usage in Python
# using the tracemalloc module.

# ---------------- WHY USE TRACEMALLOC ----------------
# tracemalloc helps you:
# ✔ Track memory allocations
# ✔ Identify memory-heavy lines of code
# ✔ Debug memory leaks

# ------------------------------------------------------------

# Import tracemalloc → used for memory tracking
import tracemalloc


# ---------------- FUNCTION ----------------
# This function creates a large list in memory
def create_list():
    
    # List comprehension creating 100,000 integers
    # This consumes memory
    return [i for i in range(100000)]


# ---------------- MAIN FUNCTION ----------------
def main():
    
    # ---------------- START MEMORY TRACKING ----------------
    # Begin tracking memory allocations
    tracemalloc.start()
    
    # Call function → memory will be allocated here
    create_list()
    
    
    # ---------------- TAKE MEMORY SNAPSHOT ----------------
    # Capture current memory allocation state
    snapshot = tracemalloc.take_snapshot()
    
    
    # ---------------- ANALYZE MEMORY USAGE ----------------
    # statistics('lineno') groups memory usage by line number
    # This helps identify which lines use the most memory
    top_stats = snapshot.statistics('lineno')

    
    # ---------------- PRINT RESULTS ----------------
    print("[ Top Memory Usage Lines ]")
    
    # Loop through memory stats
    # NOTE: You used [::] which means "all elements"
    # Usually we limit to top 10 using [:10]
    for stat in top_stats[::]:
        print(stat)


# ---------------- RUN PROGRAM ----------------
# Call main function
main()


# ================= IMPORTANT UNDERSTANDING =================
# tracemalloc works by:
# 1. Tracking memory allocations
# 2. Taking snapshots
# 3. Comparing or analyzing memory usage


# ================= OUTPUT EXPLANATION =================
# Each "stat" line shows:
# ✔ File name
# ✔ Line number
# ✔ Memory allocated
# ✔ Number of allocations

# Example:
# file.py:10: size=800 KiB, count=100000, average=8 B

# Meaning:
# Line 10 allocated ~800 KB memory
# Across 100,000 objects


# ================= IMPROVEMENT =================
# Show only top 10 lines (better practice):
#
# for stat in top_stats[:10]:
#     print(stat)


# ================= EXTRA FEATURES =================
# Compare memory snapshots:
#
# snapshot2 = tracemalloc.take_snapshot()
# top_diff = snapshot2.compare_to(snapshot1, 'lineno')
#
# Helps detect memory leaks


# ================= SIMPLE SUMMARY =================
# tracemalloc.start() → Start tracking memory
# take_snapshot() → Capture memory state
# statistics() → Analyze memory usage

[ Top Memory Usage Lines ]
d:\AI and ML\Python\.venv\Lib\site-packages\IPython\core\compilerop.py:86: size=13.5 KiB, count=140, average=98 B
d:\AI and ML\Python\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3550: size=10.8 KiB, count=2, average=5508 B
d:\AI and ML\Python\.venv\Lib\site-packages\IPython\core\compilerop.py:174: size=7149 B, count=74, average=97 B
C:\Users\tarun\AppData\Local\Programs\Python\Python313\Lib\json\decoder.py:361: size=7020 B, count=23, average=305 B
d:\AI and ML\Python\.venv\Lib\site-packages\IPython\core\async_helpers.py:145: size=5628 B, count=4, average=1407 B
d:\AI and ML\Python\.venv\Lib\site-packages\IPython\core\history.py:1044: size=5614 B, count=3, average=1871 B
d:\AI and ML\Python\.venv\Lib\site-packages\IPython\core\history.py:987: size=5506 B, count=1, average=5506 B
d:\AI and ML\Python\.venv\Lib\site-packages\jupyter_client\session.py:1061: size=3597 B, count=5, average=719 B
C:\Users\tarun\AppData\Local\Programs\Python\Python313\Lib\